In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
# helper functions that will help maintain the conversation state with claude
def add_user_message(messages, text):
    user_message = {"role":"user","content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant","content":text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params={
        "model":model,
        "max_tokens":1000,
        "messages":messages,
        "temperature":temperature
    }
    if system:
        params["system"]=system
    if stop_sequences:
        params["stop_sequences"]=stop_sequences
        
    message=client.messages.create(**params)
    
    return message.content[0].text

messages=[]

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    """
    add_user_message(messages,prompt)
    output = chat(messages)
    return output    

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    #Grading done here
    score = 10

    return{
        "output":output,
        "test_case":test_case,
        "score":score
    }

In [ ]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results=[]
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [ ]:
with open("dataset.json","r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results,indent=2))

In [ ]:
#just creating a file of results that came out of output
with open("results.json","w") as f:
    json.dump(results,f,indent=2)